In [34]:
import hashlib
import os
from pymongo import MongoClient
from pymongo.errors import DuplicateKeyError
from boltons.iterutils import remap
import json_numpy
import math

In [35]:
mongo_url = "mongodb://mongodb:65530"
# Connect to MongoDB using the URL
client = MongoClient(mongo_url)
# Access your database and collection
db = client["oscb"]
datasets_collection = db.datasets
user_datasets_collection = db.get_collection("user_datasets")
pp_results_collection = db.get_collection("pp_results")
jobs_collection = db.get_collection("jobs")
benchmarks_collection = db.get_collection("benchmarks")
bm_results_collection = db.get_collection("bm_results")
workflows_collection = db.get_collection("workflows")
large_collection = db.get_collection("large_documents")

# Chunk size (e.g., 4MB per chunk)
CHUNK_SIZE = 2 * 1024 * 1024  # 4 MB

pp_results_collection.create_index({'process_id': 1}, unique=True, background=True)
bm_results_collection.create_index({'process_id': 1}, unique=True, background=True)
workflows_collection.create_index({'workflows_id': 1}, unique=True, background=True)
datasets_collection.create_index({'Id': 1}, unique=True, background=True)
benchmarks_collection.create_index({'benchmarksId': 1}, unique=True, background=True)
jobs_collection.create_index({'job_id': 1}, unique=True, background=True)

'job_id_1'

In [37]:
def query_large_field(field_name):
    """Query and reconstruct a large field (split into 4MB chunks)."""
    pipeline = [
        {
            # Only keep docs where the field exists and is not null
            "$match": {
                field_name: {"$exists": True, "$ne": None}
            }
        },
        {
            "$project": {
                "process_id": 1,
                field_name: 1,
                "fieldSize": { "$strLenBytes": f"${field_name}" },
                # "fieldSize": {
                #     "$cond": {
                #         "if": { "$exists": f"${field_name}" },
                #         "then": { "$strLenBytes": f"${field_name}" },
                #         "else": 0  # Or handle as needed, e.g., null
                #     }
                # },
                "document_content": "$$ROOT"
            },
        },
        {"$sort": {"fieldSize": -1}},
        {"$limit": 1},
        {
            "$match": {
                "fieldSize": { "$gt": CHUNK_SIZE }
            }
        }
    ]
    
    cursor = pp_results_collection.aggregate(pipeline)
    for doc in cursor:
        document_id = f"{doc['process_id']}_{field_name}"
        chunk_string(document_id, doc[field_name])
        pp_results_collection.update_one({'process_id': doc["process_id"]}, {'$set': {field_name: f"chunked_data::{document_id}"}})
    return len(list(cursor))

In [38]:
def chunk_string(document_id, large_data):
    """Yield successive chunks from large_data."""
    # Split and insert chunks
    num_chunks = math.ceil(len(large_data) / CHUNK_SIZE)
    for i in range(num_chunks):
        chunk_data = large_data[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
        large_collection.insert_one({
            "document_id": document_id,
            "chunk_index": i,
            "data": chunk_data
        })

    print(f"Inserted {num_chunks} chunks.")

In [45]:
query_large_field("cell_metadata")

Inserted 2 chunks.


0

In [33]:
query_large_field(" highest_expr_genes")

0

In [53]:
while True:
    counts = query_large_field("umap_3D")
    if counts == 0:
        break
    # query_large_field("cell_metadata")
    # query_large_field("gene_metadata")
    # query_large_field("umap")
    # query_large_field("umap_3D")
    # query_large_field("tsne")
    # query_large_field("tsne_3D")
    # query_large_field(" highest_expr_genes")

In [ ]:
while True:
    counts = query_large_field("cell_metadata")
    if counts == 0:
        break